In [1]:
!git clone https://github.com/vngbthang/ie403-Ecom-MultiTask-Complaint-Detection.git
%cd ie403-Ecom-MultiTask-Complaint-Detection

Cloning into 'ie403-Ecom-MultiTask-Complaint-Detection'...
remote: Enumerating objects: 17351, done.
remote: Counting objects: 100% (795/795), done.
remote: Compressing objects: 100% (476/476), done.
remote: Total 17351 (delta 398), reused 701 (delta 305), pack-reused 16556 (from 2)
Receiving objects: 100% (17351/17351), 141.51 MiB | 25.07 MiB/s, done.
Resolving deltas: 100% (3591/3591), done.
/kaggle/working/ie403-Ecom-MultiTask-Complaint-Detection


In [2]:
!pip install -r requirements.txt
!pip install transformers accelerate seqeval torchcrf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 102.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=cf0a1e855d2d066d38ef293c2fb1779cde8ecf011928a70ce852015eed2b7775
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [5]:
!ls data/processed | grep uit_viocd_full_complaint_ner

uit_viocd_full_complaint_ner.json
uit_viocd_full_complaint_ner_split_summary.json
uit_viocd_full_complaint_ner_test.json
uit_viocd_full_complaint_ner_train.json
uit_viocd_full_complaint_ner_val.json


In [6]:
!python -m src.training.train_phobert_ner \
  --train-json data/processed/uit_viocd_full_complaint_ner_train.json \
  --val-json data/processed/uit_viocd_full_complaint_ner_val.json \
  --test-json data/processed/uit_viocd_full_complaint_ner_test.json \
  --output-dir outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch \
  --epochs 5 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --model-name vinai/phobert-base-v2 \
  --use-class-weights \
  --class-weight-scheme balanced \
  --disable-tqdm \
  --no-save-checkpoint

PhoBERT NER Single-Task Training
Train JSON : data/processed/uit_viocd_full_complaint_ner_train.json
Val JSON   : data/processed/uit_viocd_full_complaint_ner_val.json
Test JSON  : data/processed/uit_viocd_full_complaint_ner_test.json
Output Dir : outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch
Model Name : vinai/phobert-base-v2
Class Weights: True (balanced)
Device: cuda
[INFO] Checkpoint saving disabled by --no-save-checkpoint
config.json: 100%|█████████████████████████████| 678/678 [00:00<00:00, 2.61MB/s]
vocab.txt: 895kB [00:00, 43.4MB/s]
bpe.codes: 1.14MB [00:00, 104MB/s]
tokenizer.json: 3.13MB [00:00, 140MB/s]

[LOAD] Training data: data/processed/uit_viocd_full_complaint_ner_train.json
[LOAD] Test data: data/processed/uit_viocd_full_complaint_ner_test.json
  Train samples: 2280
  Test samples : 291

[INFO] Using class weights: scheme=balanced
  O: count=21780 weight=1.342608
  B-COMP: count=8085 weight=3.616821
  I-COMP: count=57861 weight=0.505384
pytorch_mo

In [7]:
!cat outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/metrics/phobert_ner_single_task.json

{
  "epoch": 5,
  "entity_precision": 0.7937147461724415,
  "entity_recall": 0.9044995408631772,
  "entity_f1": 0.8454935622317596,
  "token_precision_macro": 0.8683678361557682,
  "token_recall_macro": 0.8629656087727242,
  "token_f1_macro": 0.8620135631380003,
  "avg_loss": 0.24861465067717067
}

In [8]:
!head -20 outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_predictions.csv

﻿index,gold_tags,pred_tags,n_gold_tags,n_pred_tags,has_length_mismatch,correct
0,B-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP,B-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP,9,9,False,True
1,B-COMP I-COMP I-COMP,B-COMP I-COMP I-COMP,3,3,False,True
2,O O O O O O O O O O O O O B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP,O O O O O O O O O O O O O B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP,37,37,False,True
3,B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP O O O O O O O O O O O O O O O O O O O B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP O O O O O O O O O O B-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP I-COMP 

In [9]:
!zip -r uit_viocd_full_complaint_phobert_ner_weighted_5epoch_outputs.zip \
  outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch

  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/ (stored 0%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/ (stored 0%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch4.png (deflated 18%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch3.png (deflated 19%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch1.png (deflated 19%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_predictions.csv (deflated 96%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch2.png (deflated 19%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_weighted_5epoch

In [4]:
!python -m src.training.train_phobert_ner \
  --train-json data/processed/uit_viocd_full_complaint_ner_train.json \
  --val-json data/processed/uit_viocd_full_complaint_ner_val.json \
  --test-json data/processed/uit_viocd_full_complaint_ner_test.json \
  --output-dir outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch \
  --epochs 5 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --model-name vinai/phobert-base-v2 \
  --disable-tqdm \
  --no-save-checkpoint

PhoBERT NER Single-Task Training
Train JSON : data/processed/uit_viocd_full_complaint_ner_train.json
Val JSON   : data/processed/uit_viocd_full_complaint_ner_val.json
Test JSON  : data/processed/uit_viocd_full_complaint_ner_test.json
Output Dir : outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch
Model Name : vinai/phobert-base-v2
Class Weights: False (balanced)
Device: cuda
[INFO] Checkpoint saving disabled by --no-save-checkpoint
config.json: 100%|█████████████████████████████| 678/678 [00:00<00:00, 3.33MB/s]
vocab.txt: 895kB [00:00, 82.9MB/s]
bpe.codes: 1.14MB [00:00, 111MB/s]
tokenizer.json: 3.13MB [00:00, 132MB/s]

[LOAD] Training data: data/processed/uit_viocd_full_complaint_ner_train.json
[LOAD] Test data: data/processed/uit_viocd_full_complaint_ner_test.json
  Train samples: 2280
  Test samples : 291
pytorch_model.bin: 100%|██████████████████████| 540M/540M [00:03<00:00, 157MB/s]
Loading weights:  33%|▎| 65/197 [00:00<00:00, 2311.15it/s, Materializing param=

In [5]:
!cat outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/metrics/phobert_ner_single_task.json

{
  "epoch": 5,
  "entity_precision": 0.8171953255425709,
  "entity_recall": 0.898989898989899,
  "entity_f1": 0.8561434193266289,
  "token_precision_macro": 0.8824194461111631,
  "token_recall_macro": 0.8686010771264475,
  "token_f1_macro": 0.8732200110011391,
  "avg_loss": 0.21887451744262587
}

In [6]:
!zip -r uit_viocd_full_complaint_phobert_ner_unweighted_5epoch_outputs.zip \
  outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch

  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/ (stored 0%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/ (stored 0%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch5.png (deflated 19%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/phobert_ner_single_task_predictions.csv (deflated 96%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch1.png (deflated 19%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch4.png (deflated 20%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_unweighted_5epoch/figures/phobert_ner_single_task_breakdown_epoch3.png (deflated 18%)
  adding: outputs/metrics/uit_viocd_full_complaint_phobert_ner_u